# Lab 16: A reproducible scientific workspace

Build a small project for six **synthetic** samples. Keep raw inputs intact, create a report from a script, and diagnose a relative-path mistake.

Use a Python kernel on Linux/WSL or Colab. On Windows, install Git Bash first. No data downloads, package installations or network requests are needed. All writes stay inside a new temporary workspace. Run the setup once and then complete tasks in order; each `run_bash(...)` call starts at the workspace root. A `cd` persists only within that call.

The Python helper is a notebook adapter around real Bash. Write your Bash commands inside its multiline string, then use the check cells to verify the files they produce.

In [ ]:
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

git_bash = Path(os.environ.get("ProgramFiles", "C:/Program Files")) / "Git/bin/bash.exe"
bash = str(git_bash) if os.name == "nt" and git_bash.exists() else shutil.which("bash")
if bash is None:
    raise RuntimeError("Bash is required. Use Linux/WSL, Colab, or install Git Bash on Windows.")
workspace = Path(tempfile.mkdtemp(prefix="mm-linux-16-"))
raw = workspace / "data" / "raw"
raw.mkdir(parents=True)
samples = "sample_id\tcondition\treads_million\tqc_status\nS01\tcontrol\t12.4\tpass\nS02\ttreated\t18.2\tpass\nS03\tcontrol\t4.1\tfail\nS04\ttreated\t22.0\tpass\nS05\tcontrol\t15.7\tpass\nS06\ttreated\t7.3\tfail\n"
(raw / "samples.tsv").write_text(samples, encoding="utf-8", newline="\n")
original = (raw / "samples.tsv").read_bytes()
env = dict(os.environ, COURSE_PYTHON=sys.executable.replace("\\", "/"))

def run_bash(script, *, check=True):
    result = subprocess.run([bash, "-c", script], cwd=workspace, env=env,
                            text=True, capture_output=True, check=False)
    if result.stdout:
        print(result.stdout, end="")
    if result.stderr:
        print("Diagnostics:\n" + result.stderr, end="")
    if check and result.returncode:
        raise RuntimeError(f"Bash exited with status {result.returncode}")
    return result

print("Ready: six synthetic records in a fresh temporary workspace.")
print("Notebook platform:", "Colab" if IN_COLAB else "local Python")

## Task 1 — Create the layout

Use `mkdir -p` to create `scripts`, `results`, `logs` and `scratch`. List those directory names with `ls -d`. Which directory should contain an unchanged copy of the original measurements?

In [ ]:
# Create scripts, results, logs and scratch, then list the directory names.
completed = run_bash("""
mkdir -p scripts results logs scratch
ls -d scripts results logs scratch
""")

In [ ]:
assert all((workspace / name).is_dir() for name in ["scripts", "results", "logs", "scratch"]), "Create all four workspace directories."
assert (raw / "samples.tsv").read_bytes() == original, "The raw input must remain unchanged."
print("✅ Workspace directories exist and raw input is intact.")

## Task 2 — Inspect and copy the input

Print the first three lines and the line count of `data/raw/samples.tsv`. Predict the line count first. Copy it to `scratch/working.tsv`; do not rename or edit the original. Explain why the number of lines is different from the number of samples.

In [ ]:
# Preview the sample sheet, count its lines and create a working copy.
completed = run_bash("""
head -n 3 data/raw/samples.tsv
wc -l data/raw/samples.tsv
cp data/raw/samples.tsv scratch/working.tsv
""")

In [ ]:
assert (workspace / "scratch/working.tsv").read_bytes() == original, "Copy the raw sheet without changing its contents."
assert len((raw / "samples.tsv").read_text().splitlines()) == 7
print("✅ Seven lines: one header plus six sample records.")

## Task 3 — Produce a report

Use a quoted here-document to write `scripts/count_samples.py`. In Python, read `sys.argv[1]` with `csv.DictReader(..., delimiter="\t")` and print the number of records.

Run it with `"$COURSE_PYTHON" scripts/count_samples.py data/raw/samples.tsv`. The setup variable selects this notebook's interpreter explicitly. Redirect stdout to `results/sample-count.txt` and stderr to `logs/count.log`. Capture `$?` immediately, then print it. A successful report should contain `6`, and its diagnostic log should be empty.

In [ ]:
# Write a counting script and run it with separate result and diagnostic files.
completed = run_bash(r"""
cat > scripts/count_samples.py <<'PY'
import csv
import sys
with open(sys.argv[1], newline="", encoding="utf-8") as handle:
    rows = list(csv.DictReader(handle, delimiter="\t"))
print(len(rows))
PY
"$COURSE_PYTHON" scripts/count_samples.py data/raw/samples.tsv > results/sample-count.txt 2> logs/count.log
status=$?
printf 'exit status: %s\n' "$status"
cat results/sample-count.txt
""")

In [ ]:
assert (workspace / "results/sample-count.txt").read_text().strip() == "6", "The report should contain the six-record count."
assert (workspace / "logs/count.log").read_text() == "", "Inspect the log and correct any diagnostics."
assert (raw / "samples.tsv").read_bytes() == original
print("✅ The script reports six records with an empty diagnostic log.")

## Task 4 — Diagnose a path mistake

Predict why the next call fails. It is deliberately run with `check=False` so you can inspect the error without interrupting the notebook. The call changes into `results`, then tries to open a path that would only work from the project root.

In [ ]:
broken = run_bash("""cd results
head -n 1 data/raw/samples.tsv""", check=False)
print("Exit status is nonzero:", broken.returncode != 0)

Repair the input path while keeping `cd results`. Store the returned object as `repaired`; the next check reads its stdout. Remember that a new call starts at the workspace root.

In [ ]:
# Keep cd results and repair the relative path to the raw input.
repaired = run_bash("""cd results
head -n 1 ../data/raw/samples.tsv""")

In [ ]:
assert repaired.returncode == 0, "Use a path relative to results/."
assert repaired.stdout.strip() == "sample_id\tcondition\treads_million\tqc_status", "Read the original header."
print("✅ The repaired path reaches the raw input from results/.")

## Task 5 — Remove only your disposable copy

Remove the exact path `scratch/working.tsv`. Do not use a wildcard or recursive deletion. In this isolated notebook, use `rm` without the interactive `-i` flag because subprocess input is not interactive; you already know which disposable file you created.

In [ ]:
# Remove only the disposable working copy you created in Task 2.
completed = run_bash("rm scratch/working.tsv")

In [ ]:
assert not (workspace / "scratch/working.tsv").exists(), "Remove the disposable copy."
assert (raw / "samples.tsv").read_bytes() == original, "Keep the raw dataset byte-for-byte unchanged."
assert (workspace / "results/sample-count.txt").read_text().strip() == "6"
print("✅ Disposable copy removed; raw data and final report remain intact.")

## Reflection

1. Which files would a collaborator need to reproduce this report?
2. Why can a result file exist after a process fails?
3. How would you validate an unfamiliar sample sheet before counting it?
4. A `cd` in one notebook call does not persist to another. How does that differ from typing consecutive commands in one interactive terminal?

A sample count checks one property of the input. It does not establish that the samples are independent, that QC decisions are justified, or that the analysis addresses the scientific question.